# ___Fine tune SAM on a custom dataset___
--------------------------

In [1]:
!python --version

Python 3.14.2


The system cannot find the path specified.


In [2]:
import os
from PIL import Image

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# handrolled helpers
import ftune

In [2]:
# REFERENCES
# https://www.labellerr.com/blog/fine-tune-sam-on-custom-dataset/
# https://encord.com/blog/learn-how-to-fine-tune-the-segment-anything-model-sam/

In [3]:
imgs = []

for file in os.listdir(r"../images/MaxSee/"):
    with open(file=f"../images/MaxSee/{file}", mode="rb") as fp:
        # print(Image.open(fp).mode) # RGBA - damn
        # print(Image.open(fp).convert("RGB").mode) # cool :)
        imgs.append(Image.open(fp).convert("RGB"))

In [18]:
torch.Tensor(np.array([np.array(img, dtype=np.uint8) for img in imgs]))

tensor([[[[126., 139., 141.],
          [126., 139., 141.],
          [126., 140., 140.],
          ...,
          [ 91.,  91.,  94.],
          [ 91.,  91.,  96.],
          [ 91.,  91.,  96.]],

         [[125., 138., 140.],
          [125., 138., 140.],
          [125., 139., 139.],
          ...,
          [ 90.,  90.,  93.],
          [ 90.,  90.,  95.],
          [ 90.,  90.,  95.]],

         [[125., 139., 139.],
          [125., 139., 139.],
          [125., 139., 139.],
          ...,
          [ 90.,  90.,  93.],
          [ 90.,  90.,  95.],
          [ 90.,  90.,  95.]],

         ...,

         [[183., 193., 176.],
          [185., 194., 178.],
          [185., 194., 178.],
          ...,
          [169., 190., 186.],
          [173., 194., 192.],
          [176., 197., 195.]],

         [[183., 193., 176.],
          [185., 194., 178.],
          [185., 194., 178.],
          ...,
          [171., 192., 189.],
          [175., 195., 195.],
          [179., 198., 198.]],



In [27]:
def _read_images_into_tensor(fnames: list[str]) -> torch.Tensor:
    """
    :param fnames: file names of the images
    :type fnames: list[str]
    :return: a 4D tensor of shape (n_imgs, width, height, n_clrchannels)
    :rtype: Tensor
    """
    imgs: list[NDArray[np.uint8]] = []
    for fn in fnames:
        try:
            with open(file=fn, mode="rb") as fp:
                obj = Image.open(fp)  # opens in RGB colour chanel mode by default, unlike opencv, which is what we want
                if obj.mode != "RGB":
                    obj = obj.convert(r"RGB")  # if the colour channel is not RGB, convert it to RGB
                imgs.append(np.array(obj, dtype=np.uint8))
        except (PermissionError, FileNotFoundError) as excpt:
            raise RuntimeError(f"Filed to read file {fn}") from excpt
    return torch.Tensor(np.array([img for img in imgs]))

In [28]:
images = _read_images_into_tensor(fnames=[f"../images/MaxSee/{f}" for f in os.listdir(r"../images/MaxSee/")])

In [29]:
images 

False